# 18. Резервирование global test

Создаёт неизменяемый manifest без копирования данных. Новый `global_test` выбирается только из прежнего train; прежний test получает роль `legacy_test`. Этот notebook запускается один раз до переобучения моделей.

In [ ]:
from pathlib import Path
import runpy

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

PROJECT_DIR = Path('/content/drive/MyDrive/NER_RuREBus_project')
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()
runpy.run_path(str(PROJECT_DIR / 'colab_bootstrap.py'))['bootstrap_project'](PROJECT_DIR)


In [ ]:
from rurebus_ie.data.protocol_split import build_global_test_protocol, validate_global_test_protocol

SOURCE = PROJECT_DIR / 'rurebus_data/versions/corrected_v1/manifest.csv'
MANIFEST = PROJECT_DIR / 'rurebus_data/versions/corrected_v1/global_v1_manifest.csv'
REPORT = PROJECT_DIR / 'rurebus_data/versions/corrected_v1/global_v1_report.json'

if MANIFEST.exists() or REPORT.exists():
    report = validate_global_test_protocol(MANIFEST, REPORT)
    print('Протокол уже зарезервирован и прошёл проверку SHA-256.')
else:
    report = build_global_test_protocol(
        SOURCE, MANIFEST, REPORT,
        protocol_version='global_v1',
        global_test_fraction=0.15,
        validation_fraction=0.15,
        seed=20260826,
        search_trials=5000,
    )

for split, values in report['split'].items():
    print(f"{split:12s}: documents={values['documents']:3d}, groups={values['source_groups']:3d}, entities={values['entities']:5d}, relations={values['relations']:4d}")
print('Manifest:', MANIFEST)
print('SHA-256:', report['manifest_sha256'])


После этой точки `global_v1_manifest.csv` не редактируется и не пересобирается. До финального notebook разрешено использовать только `train` и `validation`.